# 🎬 Дообучение дуэта нейросетей Qwen 2.5 для вертикальных видео

Этот ноутбук дообучает две специализированные модели с помощью **Unsloth (4-bit QLoRA)** на бесплатном GPU T4 в Google Colab за ~15 минут:
1. **Режиссёр и Сценарист**: `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` на датасете вирусных шортсов (хуки, удержание, SFX, Telegram клиффхэнгеры).
2. **Оператор и Постановщик**: `unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit` на датасете визуальной непрерывности (Continuity Bible, ракурсы, Midjourney/Flux промпты).

После завершения адаптеры экспортируются в GGUF для моментального импорта в **Ollama**!

In [ ]:
# 1. Установка Unsloth и зависимостей (быстрая установка ~1 мин)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# 2. Загрузка обучающих датасетов
import json
from datasets import load_dataset

# Можно загрузить локальные файлы dataset_viral_shorts.jsonl и dataset_cinematographer.jsonl в Colab:
print("Загрузите файлы dataset_viral_shorts.jsonl и dataset_cinematographer.jsonl через панель слева в /content/")

In [ ]:
# 3. Функция дообучения одной модели
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

def train_model(model_name, dataset_file, output_lora_dir, max_steps=120):
    print(f"\n{'='*60}\nНачало обучения: {model_name}\n{'='*60}")
    max_seq_length = 2048
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    
    dataset = load_dataset("json", data_files=dataset_file, split="train")
    
    def format_prompts(batch):
        conversations = batch["messages"]
        texts = [tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False) for conv in conversations]
        return {"text": texts}
        
    dataset = dataset.map(format_prompts, batched=True)
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            max_steps=max_steps,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=42,
            output_dir=output_lora_dir,
            report_to="none",
        ),
    )
    
    trainer.train()
    print(f"Сохраняем LoRA адаптер в {output_lora_dir}...")
    model.save_pretrained(output_lora_dir)
    tokenizer.save_pretrained(output_lora_dir)
    print("Готово!")
    return model, tokenizer

In [ ]:
# 4. Дообучаем Режиссёра-Сценариста
dir_model, dir_tok = train_model(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    dataset_file="dataset_viral_shorts.jsonl",
    output_lora_dir="lora_qwen_director",
    max_steps=100
)

In [ ]:
# 5. Дообучаем Оператора-Постановщика
cin_model, cin_tok = train_model(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    dataset_file="dataset_cinematographer.jsonl",
    output_lora_dir="lora_qwen_cinematographer",
    max_steps=100
)

In [ ]:
# 6. Экспорт в GGUF для Ollama (q4_k_m)
# dir_model.save_pretrained_gguf("qwen_director_gguf", dir_tok, quantization_method="q4_k_m")
# cin_model.save_pretrained_gguf("qwen_cinematographer_gguf", cin_tok, quantization_method="q4_k_m")
print("Модели обучены! Скачайте веса lora_qwen_director и lora_qwen_cinematographer.")